In [1]:
import numpy as np
from datasets import load_dataset
from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer
import torch

In [2]:
lm1 = AutoModelForCausalLM.from_pretrained("UBC-SLIME/colx_531_smollm2-135m").to("cuda")
lm2 = AutoModelForCausalLM.from_pretrained("UBC-SLIME/colx_531_smollm2-360m").to("cuda")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/110 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/110 [00:00<?, ?B/s]

In [3]:
def compute_score(text):
    # tokenize
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to("cuda")

    # inference and no gradient updates
    with torch.no_grad():
        # determine output from each model
        outputs1 = lm1(**inputs, labels=inputs["input_ids"])
        outputs2 = lm2(**inputs, labels=inputs["input_ids"])

        # compute losses
        loss1 = outputs1.loss.item()
        loss2 = outputs2.loss.item()

    # return the difference (log-likelihood ratio score)
    return loss1 - loss2

In [4]:

# load dataset and tokenizer, extract test data
dataset = load_dataset("UBC-SLIME/colx_531_group_project")
tokenizer = AutoTokenizer.from_pretrained("UBC-SLIME/colx_531_smollm2-135m")
test_data = dataset['test']

# compute the scores
scores = [compute_score(item['text']) for item in test_data]

# determine the threshold
threshold = sum(scores) / len(scores)

# convert to binary predictions
predictions = [1 if s > threshold else 0 for s in scores]

# view results
print(f"Threshold: {threshold:.4f}")
print("First 10 scores:", [round(s, 4) for s in scores[:10]])
print("First 10 predictions:", predictions[:10])

train.json:   0%|          | 0.00/102M [00:00<?, ?B/s]

dev.json:   0%|          | 0.00/20.2M [00:00<?, ?B/s]

test.json:   0%|          | 0.00/30.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/15000 [00:00<?, ? examples/s]

tokenizer_config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Threshold: 0.1298
First 10 scores: [0.1314, 0.0727, 0.1691, 0.1202, 0.1836, 0.0318, 0.1511, 0.1096, 0.0566, 0.1263]
First 10 predictions: [1, 0, 1, 0, 1, 0, 1, 0, 0, 0]


In [5]:
import pandas as pd

# grab IDs from test data
test_ids = test_data['id']

# create dataframe for submission
submission_df = pd.DataFrame({
    'id': test_ids,
    'label': predictions
})

# export to CSV
submission_df.to_csv('submission.csv', index=False)